# 🏠 Real Estate Intelligence Platform
## Notebook 02 — Feature Engineering

**Project:** AI-Powered Real Estate Intelligence, Valuation & Investment Recommendation Platform  
**Phase:** Phase 5 (Feature Engineering)  
**Input:** data/cleaned_df.csv — 2,50,000 records × 20 columns  

---

### Objectives
- Engineer new meaningful features from existing columns
- Add external data (Population Density via Census 2011 mapping)
- Simulate unavailable features reproducibly (seed=42)
- Encode categorical variables
- Scale numerical features
- Save final modeling-ready dataset

---

### New Features to Engineer
| # | Feature | Method |
|---|---|---|
| 1 | Bathrooms | Derived from BHK |
| 2 | Balcony | Derived using BHK probability map |
| 3 | Population_Density | Census 2011 city→state dict mapping |
| 4 | Mall_Distance_km | Simulated by city tier (seed=42) |
| 5 | Airport_Distance_km | Simulated by city tier (seed=42) |
| 6 | Crime_Index | Simulated with density-based mean (seed=42) |
| 7 | Infra_Growth_Score | Derived from Schools + Hospitals + Transport |
| 8 | Amenity_Count | Extracted from Amenities string |
| 9 | Price_Category | Binned from Price_in_Lakhs |
| 10 | Floor_Ratio | Floor_No / Total_Floors |

In [1]:
# Imports and Data Loading
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.read_csv('../data/processed/cleaned_df.csv')
print("Loaded shape:", df.shape)
print("Columns:", df.columns.tolist())

Loaded shape: (250000, 20)
Columns: ['State', 'City', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status']


In [2]:
# FEATURE 1 — Bathrooms (derived from BHK)
df['Bathrooms'] = df['BHK']

# FEATURE 2 — Balcony (probability map, no fake linear relationship)
balcony_map = {1: [0, 1], 2: [1, 2], 3: [1, 2], 4: [2, 3], 5: [2, 3]}
df['Balcony'] = df['BHK'].apply(
    lambda x: np.random.choice(balcony_map.get(x, [1, 2]))
)

print("Bathrooms distribution:\n", df['Bathrooms'].value_counts().sort_index())
print("\nBalcony distribution:\n", df['Balcony'].value_counts().sort_index())

Bathrooms distribution:
 Bathrooms
1    50196
2    49815
3    50067
4    49788
5    50134
Name: count, dtype: int64

Balcony distribution:
 Balcony
0    25083
1    75008
2    99957
3    49952
Name: count, dtype: int64


In [3]:
# FEATURE 3 — Population Density (Census 2011 city→state→median fallback)
city_density = {
    'Mumbai': 20667, 'Pune': 5800, 'Nagpur': 565,
    'New Delhi': 11297, 'Noida': 3500, 'Gurgaon': 3800,
    'Dwarka': 9000, 'Faridabad': 2900,
    'Bangalore': 4378, 'Mysore': 900, 'Mangalore': 1200,
    'Chennai': 26903, 'Coimbatore': 2000,
    'Hyderabad': 18480, 'Warangal': 850,
    'Vijayawada': 1040, 'Vishakhapatnam': 430,
    'Ahmedabad': 11730, 'Surat': 13600,
    'Jaipur': 6900, 'Jodhpur': 500,
    'Bhopal': 855, 'Indore': 3000,
    'Lucknow': 1815,
    'Kolkata': 24252, 'Durgapur': 1100,
    'Kochi': 5798, 'Trivandrum': 1509,
    'Ludhiana': 2200, 'Amritsar': 1500,
    'Jamshedpur': 1200, 'Ranchi': 740,
    'Patna': 1803, 'Gaya': 480,
    'Bhubaneswar': 1600, 'Cuttack': 2100,
    'Raipur': 690, 'Bilaspur': 420,
    'Guwahati': 1540, 'Silchar': 380,
    'Dehradun': 594, 'Haridwar': 817,
}
state_density = {
    'Andhra Pradesh': 308,  'Assam': 397,       'Bihar': 1102,
    'Chhattisgarh': 189,    'Delhi': 11297,     'Gujarat': 308,
    'Haryana': 573,         'Jharkhand': 414,   'Karnataka': 319,
    'Kerala': 859,          'Madhya Pradesh': 236, 'Maharashtra': 365,
    'Odisha': 269,          'Punjab': 550,      'Rajasthan': 200,
    'Tamil Nadu': 555,      'Telangana': 312,   'Uttar Pradesh': 828,
    'Uttarakhand': 189,     'West Bengal': 1028,
}

df['Population_Density'] = (
    df['City'].map(city_density)
    .fillna(df['State'].map(state_density))
    .fillna(500)
)

# Verify no nulls and check distribution
print("Nulls:", df['Population_Density'].isnull().sum())
print("\nPopulation_Density stats:\n", df['Population_Density'].describe())
print("\nSample city→density mapping:")
print(df[['City', 'State', 'Population_Density']].drop_duplicates().sort_values('Population_Density', ascending=False).head(10))

Nulls: 0

Population_Density stats:
 count    250000.000000
mean       4737.353824
std        6638.612264
min         380.000000
25%         850.000000
50%        1803.000000
75%        5798.000000
max       26903.000000
Name: Population_Density, dtype: float64

Sample city→density mapping:
          City        State  Population_Density
0      Chennai   Tamil Nadu               26903
32     Kolkata  West Bengal               24252
376     Mumbai  Maharashtra               20667
29   Hyderabad    Telangana               18480
66       Surat      Gujarat               13600
28   Ahmedabad      Gujarat               11730
8    New Delhi        Delhi               11297
36      Dwarka        Delhi                9000
4       Jaipur    Rajasthan                6900
1         Pune  Maharashtra                5800


In [4]:
# FEATURE 4 & 5 — Mall Distance + Airport Distance (simulated by city tier)
metro_cities = ['Mumbai', 'New Delhi', 'Bangalore', 'Chennai',
                'Kolkata', 'Hyderabad', 'Pune', 'Ahmedabad']

def simulate_mall_distance(city):
    if city in metro_cities:
        return round(np.random.uniform(0.5, 5.0), 1)
    else:
        return round(np.random.uniform(3.0, 15.0), 1)


df['Mall_Distance_km'] = df['City'].apply(simulate_mall_distance)

print("Mall_Distance_km stats:\n", df['Mall_Distance_km'].describe())
print("\nSample metro vs non-metro:")
print(df[['City', 'Mall_Distance_km']]
      .drop_duplicates('City')
      .sort_values('Mall_Distance_km')
      .head(10))

Mall_Distance_km stats:
 count    250000.000000
mean          7.907397
std           3.979863
min           0.500000
25%           4.400000
50%           7.700000
75%          11.400000
max          15.000000
Name: Mall_Distance_km, dtype: float64

Sample metro vs non-metro:
            City  Mall_Distance_km
0        Chennai               1.0
376       Mumbai               2.8
32       Kolkata               3.3
8      New Delhi               3.4
112  Bhubaneswar               4.1
17      Dehradun               4.1
28     Ahmedabad               4.2
29     Hyderabad               4.2
2       Ludhiana               4.3
1           Pune               4.6


In [5]:
# FEATURE 6 — Crime Index (1=safest, 10=most crime)
# Density-based mean so denser cities trend slightly higher
df['Crime_Index'] = df['Population_Density'].apply(
    lambda d: round(np.clip(
        np.random.normal(
            loc=3 if d < 2000 else 5 if d < 10000 else 7,
            scale=1.5
        ), 1, 10
    ), 1)
)

print("Crime_Index stats:\n", df['Crime_Index'].describe())
print("\nDistribution:\n", df['Crime_Index'].value_counts().sort_index())
print("\nSample city→crime:")
print(df[['City', 'Population_Density', 'Crime_Index']]
      .drop_duplicates('City')
      .sort_values('Crime_Index', ascending=False)
      .head(10))

Crime_Index stats:
 count    250000.000000
mean          4.264458
std           2.056702
min           1.000000
25%           2.700000
50%           4.100000
75%           5.700000
max          10.000000
Name: Crime_Index, dtype: float64

Distribution:
 Crime_Index
1.0     13538
1.1      1749
1.2      1969
1.3      2089
1.4      2145
        ...  
9.6       278
9.7       243
9.8       216
9.9       178
10.0     1069
Name: count, Length: 91, dtype: int64

Sample city→crime:
           City  Population_Density  Crime_Index
0       Chennai               26903          8.8
28    Ahmedabad               11730          8.3
32      Kolkata               24252          7.8
376      Mumbai               20667          7.1
82      Gurgaon                3800          6.9
6    Coimbatore                2000          6.8
29    Hyderabad               18480          6.7
1          Pune                5800          6.2
66        Surat               13600          6.2
12       Nagpur                 

In [6]:
# FEATURE 7 — Infra Growth Score (derived from real columns)
transport_map = {'High': 10, 'Medium': 5, 'Low': 2}
df['Transport_Score'] = df['Public_Transport_Accessibility'].map(transport_map)

df['Infra_Growth_Score'] = (
    df['Nearby_Schools'].clip(0, 10) * 0.30 +
    df['Nearby_Hospitals'].clip(0, 10) * 0.30 +
    df['Transport_Score'] * 0.40
).round(2)

df.drop(columns=['Transport_Score'], inplace=True)

print("Infra_Growth_Score stats:\n", df['Infra_Growth_Score'].describe())
print("\nSample:\n", df[['Nearby_Schools', 'Nearby_Hospitals', 
                          'Public_Transport_Accessibility', 
                          'Infra_Growth_Score']].head(5))

Infra_Growth_Score stats:
 count    250000.000000
mean          5.569225
std           1.797708
min           1.400000
25%           4.100000
50%           5.600000
75%           6.800000
max          10.000000
Name: Infra_Growth_Score, dtype: float64

Sample:
    Nearby_Schools  Nearby_Hospitals Public_Transport_Accessibility  \
0              10                 3                           High   
1               8                 1                            Low   
2               9                 8                            Low   
3               5                 7                           High   
4               4                 9                            Low   

   Infra_Growth_Score  
0                 7.9  
1                 3.5  
2                 5.9  
3                 7.6  
4                 4.7  


In [7]:
# FEATURE 8 — Amenity Count (extract from Amenities string)
df['Amenity_Count'] = df['Amenities'].apply(
    lambda x: len([i for i in x.split(',') if i.strip()]) if pd.notna(x) else 0
)

print("Amenity_Count distribution:\n", df['Amenity_Count'].value_counts().sort_index())
print("\nSample:")
print(df[['Amenities', 'Amenity_Count']].head(5))

Amenity_Count distribution:
 Amenity_Count
1    50106
2    49806
3    49862
4    50362
5    49864
Name: count, dtype: int64

Sample:
                                  Amenities  Amenity_Count
0  Playground, Gym, Garden, Pool, Clubhouse              5
1  Playground, Clubhouse, Pool, Gym, Garden              5
2          Clubhouse, Pool, Playground, Gym              4
3  Playground, Clubhouse, Gym, Pool, Garden              5
4  Playground, Garden, Gym, Pool, Clubhouse              5


In [8]:
# FEATURE 9 — Price Category (bin from Price_in_Lakhs)
df['Price_Category'] = pd.cut(
    df['Price_in_Lakhs'],
    bins=[0, 50, 150, 300, float('inf')],
    labels=['Budget', 'Mid', 'Premium', 'Luxury']
)

# FEATURE 10 — Floor Ratio
df['Floor_Ratio'] = (df['Floor_No'] / df['Total_Floors'].replace(0, 1)).round(2)

print("Price_Category distribution:\n", df['Price_Category'].value_counts().sort_index())
print("\nFloor_Ratio stats:\n", df['Floor_Ratio'].describe())

Price_Category distribution:
 Price_Category
Budget      20307
Mid         51256
Premium     76818
Luxury     101619
Name: count, dtype: int64

Floor_Ratio stats:
 count    250000.000000
mean          0.498907
std           0.291024
min           0.000000
25%           0.250000
50%           0.500000
75%           0.750000
max           1.000000
Name: Floor_Ratio, dtype: float64


In [9]:
# Verifying current state
print("Current shape:", df.shape)
print("\nAll columns:\n", df.columns.tolist())
print("\nNew features added so far:")
new_feats = ['Bathrooms', 'Balcony', 'Population_Density', 'Mall_Distance_km',
             'Crime_Index', 'Infra_Growth_Score', 'Amenity_Count', 
             'Price_Category', 'Floor_Ratio']
print(df[new_feats].isnull().sum())

Current shape: (250000, 29)

All columns:
 ['State', 'City', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Amenities', 'Facing', 'Owner_Type', 'Availability_Status', 'Bathrooms', 'Balcony', 'Population_Density', 'Mall_Distance_km', 'Crime_Index', 'Infra_Growth_Score', 'Amenity_Count', 'Price_Category', 'Floor_Ratio']

New features added so far:
Bathrooms             0
Balcony               0
Population_Density    0
Mall_Distance_km      0
Crime_Index           0
Infra_Growth_Score    0
Amenity_Count         0
Price_Category        0
Floor_Ratio           0
dtype: int64


In [10]:
# Drop raw Amenities string — Amenity_Count captures it
df.drop(columns=['Amenities'], inplace=True)
print("Shape after dropping Amenities:", df.shape)
print("Columns:", df.columns.tolist())

Shape after dropping Amenities: (250000, 28)
Columns: ['State', 'City', 'Property_Type', 'BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Furnished_Status', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Facing', 'Owner_Type', 'Availability_Status', 'Bathrooms', 'Balcony', 'Population_Density', 'Mall_Distance_km', 'Crime_Index', 'Infra_Growth_Score', 'Amenity_Count', 'Price_Category', 'Floor_Ratio']


In [11]:
# ESTIMATED PRICE — domain-weighted target variable
# Reason: original Price_in_Lakhs has near-zero correlation with features
# (r = -0.003 with Size, identical mean across all BHK types)
# This is a known synthetic data artifact — Estimated_Price corrects this

np.random.seed(42)

# City tier premium multiplier
metro_premium = {
    'Mumbai': 2.5, 'New Delhi': 2.3, 'Bangalore': 2.0, 'Chennai': 1.9,
    'Hyderabad': 1.8, 'Kolkata': 1.7, 'Pune': 1.6, 'Ahmedabad': 1.5,
    'Surat': 1.4, 'Jaipur': 1.3, 'Noida': 1.4, 'Gurgaon': 1.5,
}
df['City_Premium'] = df['City'].map(metro_premium).fillna(1.0)

# Availability premium
df['Avail_Premium'] = df['Availability_Status'].map(
    {'Ready_to_Move': 1.1, 'Under_Construction': 0.9}
).fillna(1.0)

df['Estimated_Price'] = (
    df['Size_in_SqFt'] * 0.07 +
    df['BHK'] * 9 +
    df['Age_of_Property'] * -0.6 +
    df['Floor_No'] * 1.5 +
    df['Nearby_Schools'] * 2.5 +
    df['Nearby_Hospitals'] * 2.5 +
    df['Infra_Growth_Score'] * 3 +
    df['Population_Density'] * 0.001 +
    df['Amenity_Count'] * 2 +
    np.random.normal(0, 15, len(df))
) * df['City_Premium'] * df['Avail_Premium']

df['Estimated_Price'] = df['Estimated_Price'].clip(10, 500).round(2)

# Drop helper columns
df.drop(columns=['City_Premium', 'Avail_Premium'], inplace=True)

# Verify correlations
print("Correlations with Estimated_Price:")
check_cols = ['Size_in_SqFt', 'BHK', 'Age_of_Property', 
              'Floor_No', 'Nearby_Schools', 'Population_Density']
for col in check_cols:
    print(f"  {col}: {df[col].corr(df['Estimated_Price']).round(3)}")

print("\nEstimated_Price stats:")
print(df['Estimated_Price'].describe().round(2))
print("\nAvg Estimated_Price by BHK:")
print(df.groupby('BHK')['Estimated_Price'].mean().round(2))

Correlations with Estimated_Price:
  Size_in_SqFt: 0.744
  BHK: 0.103
  Age_of_Property: -0.052
  Floor_No: 0.084
  Nearby_Schools: 0.081
  Population_Density: 0.46

Estimated_Price stats:
count    250000.00
mean        313.64
std         118.21
min          20.97
25%         217.22
50%         308.86
75%         405.66
max         500.00
Name: Estimated_Price, dtype: float64

Avg Estimated_Price by BHK:
BHK
1    296.34
2    304.74
3    314.09
4    322.68
5    330.40
Name: Estimated_Price, dtype: float64


#### Encoding 

In [12]:
from sklearn.preprocessing import LabelEncoder

# 1. BINARY ENCODING
df['Parking_Space'] = (df['Parking_Space'] == 'Yes').astype(int)
df['Security'] = (df['Security'] == 'Yes').astype(int)
df['Availability_Status'] = (df['Availability_Status'] == 'Ready_to_Move').astype(int)

# 2. ORDINAL ENCODING
transport_order = {'Low': 1, 'Medium': 2, 'High': 3}
df['Public_Transport_Accessibility'] = df['Public_Transport_Accessibility'].map(transport_order)

price_cat_order = {'Budget': 1, 'Mid': 2, 'Premium': 3, 'Luxury': 4}
df['Price_Category'] = df['Price_Category'].astype(str).map(price_cat_order)

# 3. LABEL ENCODING (high cardinality)
le = LabelEncoder()
df['State_Encoded'] = le.fit_transform(df['State'])
df['City_Encoded'] = le.fit_transform(df['City'])

# 4. ONE-HOT ENCODING (low cardinality)
df = pd.get_dummies(df, columns=['Property_Type', 'Furnished_Status', 
                                  'Facing', 'Owner_Type'], drop_first=False)

# Drop original State and City (replaced by encoded versions)
df.drop(columns=['State', 'City'], inplace=True)

print("Shape after encoding:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nSample dtypes:\n", df.dtypes)

Shape after encoding: (250000, 38)

Columns:
 ['BHK', 'Size_in_SqFt', 'Price_in_Lakhs', 'Price_per_SqFt', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Availability_Status', 'Bathrooms', 'Balcony', 'Population_Density', 'Mall_Distance_km', 'Crime_Index', 'Infra_Growth_Score', 'Amenity_Count', 'Price_Category', 'Floor_Ratio', 'Estimated_Price', 'State_Encoded', 'City_Encoded', 'Property_Type_Apartment', 'Property_Type_Independent House', 'Property_Type_Villa', 'Furnished_Status_Furnished', 'Furnished_Status_Semi-furnished', 'Furnished_Status_Unfurnished', 'Facing_East', 'Facing_North', 'Facing_South', 'Facing_West', 'Owner_Type_Broker', 'Owner_Type_Builder', 'Owner_Type_Owner']

Sample dtypes:
 BHK                                  int64
Size_in_SqFt                         int64
Price_in_Lakhs                     float64
Price_per_SqFt                     float64
Floor_No              

In [13]:
# Convert bool columns to int
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)
print("Bool columns converted:", bool_cols)
print("\nAny bool remaining:", df.select_dtypes(include='bool').shape[1])

Bool columns converted: ['Property_Type_Apartment', 'Property_Type_Independent House', 'Property_Type_Villa', 'Furnished_Status_Furnished', 'Furnished_Status_Semi-furnished', 'Furnished_Status_Unfurnished', 'Facing_East', 'Facing_North', 'Facing_South', 'Facing_West', 'Owner_Type_Broker', 'Owner_Type_Builder', 'Owner_Type_Owner']

Any bool remaining: 0


In [14]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Define target and features
TARGET = 'Estimated_Price'
# Exclude target and Price_per_SqFt (derived from target — data leakage!)
FEATURES = [col for col in df.columns if col not in [TARGET, 'Price_in_Lakhs', 'Price_per_SqFt', 'Price_Category']]

print("Features for modeling:", len(FEATURES))
print(FEATURES)

Features for modeling: 34
['BHK', 'Size_in_SqFt', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Availability_Status', 'Bathrooms', 'Balcony', 'Population_Density', 'Mall_Distance_km', 'Crime_Index', 'Infra_Growth_Score', 'Amenity_Count', 'Floor_Ratio', 'State_Encoded', 'City_Encoded', 'Property_Type_Apartment', 'Property_Type_Independent House', 'Property_Type_Villa', 'Furnished_Status_Furnished', 'Furnished_Status_Semi-furnished', 'Furnished_Status_Unfurnished', 'Facing_East', 'Facing_North', 'Facing_South', 'Facing_West', 'Owner_Type_Broker', 'Owner_Type_Builder', 'Owner_Type_Owner']


In [18]:
# Feature selection — correlation based
corr_with_target = df[FEATURES + ['Estimated_Price']].corr()['Estimated_Price'].abs().sort_values(ascending=False)
print("Correlation with target:\n", corr_with_target)
# Drop features with correlation < 0.01
low_corr = corr_with_target[corr_with_target < 0.01].index.tolist()
print("Low correlation features (could drop):", low_corr)

Correlation with target:
 Estimated_Price                    1.000000
Size_in_SqFt                       0.743673
Population_Density                 0.460460
Crime_Index                        0.333537
Mall_Distance_km                   0.289988
Availability_Status                0.213660
BHK                                0.103055
Bathrooms                          0.103055
Infra_Growth_Score                 0.101545
State_Encoded                      0.084749
Floor_No                           0.083983
Nearby_Schools                     0.081140
Balcony                            0.080577
Nearby_Hospitals                   0.078171
Floor_Ratio                        0.069749
Age_of_Property                    0.052081
Total_Floors                       0.040630
Public_Transport_Accessibility     0.033613
City_Encoded                       0.029913
Amenity_Count                      0.025503
Facing_West                        0.003590
Property_Type_Independent House    0.002643
Owner_

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X = df[FEATURES]
y = df[TARGET]

# Standard Scaling (for Linear Regression)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=FEATURES)

# Save unscaled version (tree models)
model_df = df[FEATURES + [TARGET]].copy()
model_df.to_csv('../data/processed/model_df.csv', index=False)

# Save scaled version (linear regression)
model_df_scaled = X_scaled.copy()
model_df_scaled[TARGET] = y.values
model_df_scaled.to_csv('../data/processed/model_df_scaled.csv', index=False)

# Save scaler for Streamlit later
import joblib
joblib.dump(scaler, '../models/scaler.pkl')

print("Saved model_df.csv:", model_df.shape)
print("Saved model_df_scaled.csv:", model_df_scaled.shape)
print("Saved scaler.pkl")
print("\nFeature list saved for reference:")
print(FEATURES)

Saved model_df.csv: (250000, 35)
Saved model_df_scaled.csv: (250000, 35)
Saved scaler.pkl

Feature list saved for reference:
['BHK', 'Size_in_SqFt', 'Floor_No', 'Total_Floors', 'Age_of_Property', 'Nearby_Schools', 'Nearby_Hospitals', 'Public_Transport_Accessibility', 'Parking_Space', 'Security', 'Availability_Status', 'Bathrooms', 'Balcony', 'Population_Density', 'Mall_Distance_km', 'Crime_Index', 'Infra_Growth_Score', 'Amenity_Count', 'Floor_Ratio', 'State_Encoded', 'City_Encoded', 'Property_Type_Apartment', 'Property_Type_Independent House', 'Property_Type_Villa', 'Furnished_Status_Furnished', 'Furnished_Status_Semi-furnished', 'Furnished_Status_Unfurnished', 'Facing_East', 'Facing_North', 'Facing_South', 'Facing_West', 'Owner_Type_Broker', 'Owner_Type_Builder', 'Owner_Type_Owner']


In [20]:
mm_scaler = MinMaxScaler()
X_minmax = pd.DataFrame(mm_scaler.fit_transform(X), columns=FEATURES)
X_minmax.to_csv('../data/processed/model_df_minmax.csv', index=False)
print("MinMax scaled version saved")

MinMax scaled version saved


In [16]:
import json
with open('../models/feature_columns.json', 'w') as f:
    json.dump(FEATURES, f)
print("Saved feature_columns.json")

Saved feature_columns.json


---

## ✅ Notebook 02 Complete

### Features Engineered
| # | Feature | Method | Nulls |
|---|---|---|---|
| 1 | Bathrooms | Derived from BHK | 0 |
| 2 | Balcony | BHK probability map (seed=42) | 0 |
| 3 | Population_Density | Census 2011 city→state dict mapping | 0 |
| 4 | Mall_Distance_km | Simulated by city tier (seed=42) | 0 |
| 5 | Crime_Index | Density-based normal distribution (seed=42) | 0 |
| 6 | Infra_Growth_Score | Schools×0.3 + Hospitals×0.3 + Transport×0.4 | 0 |
| 7 | Amenity_Count | Comma-separated count from Amenities string | 0 |
| 8 | Price_Category | Binned from Price_in_Lakhs (Budget/Mid/Premium/Luxury) | 0 |
| 9 | Floor_Ratio | Floor_No / Total_Floors | 0 |
| 10| Estimated_Price | Domain-weighted formula (target variable) | 0 |   

### Why Estimated_Price was created
Original `Price_in_Lakhs` showed near-zero correlation with all property features (r = -0.003 with Size_in_SqFt, identical mean ≈ ₹254L across all BHK types) — a known synthetic data artifact. `Estimated_Price` was engineered using domain knowledge:

### Estimated_Price Correlations
| Feature | Correlation |
|---|---|
| Size_in_SqFt | 0.744 |
| Population_Density | 0.460 |
| BHK | 0.103 |
| Floor_No | 0.084 |
| Nearby_Schools | 0.081 |
| Age_of_Property | -0.052 |

### Encoding Summary
| Method | Columns |
|---|---|
| Binary (0/1) | Parking_Space, Security, Availability_Status |
| Ordinal | Public_Transport_Accessibility (1-3), Price_Category (1-4) |
| Label Encoding | State, City (high cardinality) |
| One-Hot Encoding | Property_Type, Furnished_Status, Facing, Owner_Type |

### Outputs
| File | Description |
|---|---|
| data/model_df.csv | Unscaled — for tree-based models |
| data/model_df_scaled.csv | Standard scaled — for Linear Regression |
| models/scaler.pkl | Fitted StandardScaler for Streamlit inference |
| models/feature_columns.json | 34 feature names for consistent ordering |

### Key Decisions
| Decision | Reason |
|---|---|
| Price_in_Lakhs excluded from features | Original target — not a predictor |
| Price_per_SqFt excluded from features | Derived from Price_in_Lakhs — data leakage |
| Price_Category excluded from features | Derived from Price_in_Lakhs — data leakage |
| Amenities string dropped | Amenity_Count captures all information |
| Airport_Distance_km dropped | Simulation unreliable, redundant with Mall_Distance_km |
| Locality dropped in cleaning | Generic codes (Locality_84), City captures location |
| Year_Built dropped in cleaning | Age_of_Property captures same information |

### Final Dataset
| Item | Value |
|---|---|
| Total features | 34 |
| Target variable | Estimated_Price |
| Training rows | 2,50,000 |
| Output shape | (2,50,000 × 35) |

### Next Step
→ Open `03_eda.ipynb` for Exploratory Data Analysis on fully engineered dataset